<div style="background:#3366FF; color:white; padding:12px; box-sizing:border-box; border-radius:4px;">

</div>

# Peru GDP Real-Time Dataset


> **Author:** Jason Cruz  
  **Last updated:** 11/13/2025  
  **Python version:** 3.12  
  **Project:** Rationality and Nowcasting on Peruvian GDP Revisions 

---

## Summary
Welcome to the Peru GDP Real-Time Dataset (RTD) construction notebook. This notebook walks through the end-to-end pipeline for building real-time GDP revisions from the Central Reserve Bank of Peru (BCRP) Weekly Reports.

What this notebook covers:
1. Download PDFs from BCRP Weekly Reports.
2. Create shortened PDFs with key GDP tables.
3. Clean tables and build vintages for Table 1 and Table 2.
4. Concatenate vintages into RTDs.
5. Update metadata and generate base-year and benchmark datasets.
6. Convert RTDs to releases datasets.

Main data source: BCRP Weekly Report (https://www.bcrp.gob.pe/publicaciones/nota-semanal.html)
For questions, contact: Jason Cruz (jj.cruza@up.edu.pe)

---


### Initial Set-up

Before running the pipeline:

1. Load configuration from config/config.yaml.
2. Import pipeline modules from peru_gdp_rtd.
3. Create required folders for data, metadata, records, and alerts.


> 🚧 Although the second step (database connection) is pending, the notebook currently works using **flat files (CSV)**. These CSV files will **not be saved in GitHub** as they are included in the `.gitignore` to ensure no data is stored publicly. Users can be confident that no data will be stored on GitHub. The notebook **automatically generates the CSV files**, giving users direct access to the dataset on their own systems. The data is created on the fly and can be saved locally for further use.

### Import pipeline modules


This notebook uses functions from the peru_gdp_rtd package. Make sure dependencies are installed and config/config.yaml is available.


In [1]:
# ========================================
# PERU GDP RTD - Notebook Imports
# ========================================
from pathlib import Path
import os
import pandas as pd

from peru_gdp_rtd.config import get_settings
from peru_gdp_rtd.scrapers.bcrp_scraper import pdf_downloader
from peru_gdp_rtd.processors.file_organizer import organize_files_by_year, replace_defective_pdfs
from peru_gdp_rtd.processors.pdf_processor import pdf_input_generator
from peru_gdp_rtd.orchestration.runners import build_table_1_vintages, build_table_2_vintages
from peru_gdp_rtd.transformers.concatenator import concatenate_table_1, concatenate_table_2
from peru_gdp_rtd.transformers.metadata_handler import (
    update_metadata,
    apply_base_year_sentinel,
    convert_to_benchmark_dataset,
)
from peru_gdp_rtd.transformers.releases_converter import convert_to_releases_dataset

print("Modules imported successfully.")


pygame 2.5.2 (SDL 2.28.3, Python 3.12.1)
Hello from the pygame community. https://www.pygame.org/contribute.html


Before you begin, install the required libraries listed in requirements.txt and requirements-dev.txt.


In [2]:
# If you need to install dependencies inside the notebook:
# !pip install -r ../requirements.txt


**Check out Python information**

In [3]:
import sys
import platform

print("Python Information")
print(f"  Version  : {sys.version.split()[0]}")
print(f"  Compiler : {platform.python_compiler()}")
print(f"  Build    : {platform.python_build()}")
print(f"  OS       : {platform.system()} {platform.release()}")


🐍 Python Information
  Version  : 3.12.1
  Compiler : MSC v.1916 64 bit (AMD64)
  Build    : ('main', 'Jan 19 2024 15:44:08')
  OS       : Windows 10


### Create necessary folders


We will start by creating the necessary folders to store the data at various stages of processing. The following code ensures all required directories exist, and if not, it creates them.

In [ ]:
from pathlib import Path

settings = get_settings("config/config.yaml", force_reload=True)

pdf_raw = Path(settings.paths.pdf_raw)
pdf_shortened = Path(settings.paths.pdf_input)
data_input = Path(settings.paths.data_input)
data_vintages = Path(settings.paths.vintages)
data_releases = Path(settings.paths.releases)
metadata_folder = Path(settings.paths.metadata)
record_folder = Path(settings.paths.record)
alert_track_folder = Path(settings.paths.alert_track)

for folder in [
    pdf_raw,
    pdf_shortened,
    data_input,
    data_vintages,
    data_releases,
    metadata_folder,
    record_folder,
    alert_track_folder,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Paths initialized.")
print(f"Raw PDFs: {pdf_raw}")
print(f"Shortened PDFs: {pdf_shortened}")
print(f"Vintages output: {data_vintages}")
print(f"Releases output: {data_releases}")


---

## Step 1: Download PDFs


---

The BCRP Weekly Report is the primary source for constructing the Peru GDP Real-Time Dataset (RTD). The report contains two key tables:

- Table 1: Monthly GDP growth rates (12-month percentage changes)
- Table 2: Quarterly and annual GDP growth rates (12-month percentage changes)

This step downloads new PDFs from the official BCRP site, skips files already recorded in record/1_downloaded_pdfs.txt, and organizes downloads by year.

What the scraper does:

1. Opens the official BCRP Weekly Report page.
2. Collects one PDF per month and downloads in chronological order (oldest to newest).
3. Optionally plays alerts after each batch.
4. Organizes PDFs into year-based folders.

Notes:
- If a CAPTCHA appears, solve it in the browser and re-run the scraper.
- webdriver-manager handles browser drivers automatically.
- To enable audio alerts, place an MP3 file in alert_track/.


### Download PDFs from BCRP


In [ ]:
pdf_downloader(
    settings=settings,
    max_downloads=settings.scraper.max_downloads,
    downloads_per_batch=settings.scraper.downloads_per_batch,
    headless=settings.scraper.headless,
)


### Organize downloaded PDFs

After downloading, organize PDFs into year-based folders to keep the raw data structured.


In [ ]:
organize_files_by_year(str(pdf_raw))


### Handle defective PDFs

If a PDF is corrupted or incomplete, replace it with a valid copy. Provide the year folder, the defective file name, and the replacement code.


In [ ]:
replace_defective_pdfs(
    items=[
        ("2017", "ns-08-2017.pdf", "ns-07-2017"),
        ("2019", "ns-23-2019.pdf", "ns-22-2019"),
    ],
    root_folder=str(pdf_raw),
    record_folder=str(record_folder),
    download_record_txt="1_downloaded_pdfs.txt",
    quarantine=str(pdf_raw / "_quarantine"),
)


> ⚡ **Troubleshooting Tip:** If you encounter any issues during the data cleansing step (section 3), and suspect that the problem lies with defective PDFs, you can replace those PDFs using the above function. This will help avoid errors in the following sections. In case you encounter a problem with any particular defective PDF, you can also download alternative versions of the Weekly Reports for the same month, and replace the faulty ones as needed.

#### 🧩 Key Takeaways
- Downloading PDFs: The scraper bot automates the process of collecting the latest BCRP Weekly Reports.
- Organizing PDFs: After downloading, the PDFs are organized by year to make further processing easier.
- Replacing Defective PDFs: If any PDFs are corrupted or incomplete, you can replace them with valid ones to ensure clean data.

> 🚀 **Next Steps**: With the PDFs downloaded, organized, and ready for use, we can move on to the data cleaning and extraction steps. This will be covered in the next section of the notebook. 

---

## Step 2: Shorten PDFs with key tables


---

Each Weekly Report PDF contains many pages, but only a few are needed. In this step we extract the cover page and the pages containing Table 1 and Table 2, producing shortened PDFs stored in data/raw/new_weekly_reports/shortened_pdfs/<year>/.

How it works:
1. Search for keyword pages (for example, ECONOMIC SECTORS).
2. Create a shortened PDF with those pages.
3. If a 4-page shortened file is produced, keep pages 1 and 3 (key tables).
4. Track processed files in record/2_shortened_pdfs.txt to avoid reprocessing.

This step uses PyMuPDF for text search and pypdf for page manipulation.


### Run the code to generate shortened PDFs

The function below creates shortened PDFs with the key GDP tables.


In [ ]:
pdf_input_generator(
    settings=settings,
    keywords=settings.pdf_processing.keywords,
    interactive=False,
    verbose=True,
    force=False,
)


This code processes the raw WR PDFs, extracts the pages containing the key tables, and stores them in the shortened PDF folder.


### Organize shortened PDFs

After shortening, organize PDFs into year-based subfolders for easier access.


In [ ]:
organize_files_by_year(str(pdf_shortened))


This will ensure that each trimmed WR PDF is placed into its respective year folder, making it simple to access data from specific years.

#### Moving forward

With shortened PDFs ready, proceed to table extraction and vintage creation.


#### Key takeaways

- Shortened PDFs contain only the pages needed for GDP tables.
- Files are organized by year.
- Record tracking avoids reprocessing.


> 🚀 **Next Steps:** Now that we have our trimmed PDFs, we are ready to move on to the data extraction and cleaning steps, where we will begin working with the key data from these PDFs.

---

## Step 3: Clean tables and build vintages


---

This step extracts and cleans the GDP tables, then converts them to vintage format. It handles both old CSV sources (pre-2013) and new shortened PDFs (2013+). The notebook calls build_table_1_vintages and build_table_2_vintages, which handle extraction, cleaning, and saving to data/input/table_1/ and data/input/table_2/.

Table extraction uses tabula-py where needed. Cleaning uses the functions in peru_gdp_rtd.cleaners, and the vintage format is produced by peru_gdp_rtd.transformers.VintagesPreparator.


### Main Issues in Weekly Reports and How We Cleaned Them

The BCRP Weekly Reports often present structural inconsistencies that can complicate data extraction. In this section, we focus on resolving key issues that commonly arise, ensuring the data is consistent, usable, and ready for analysis.

Below are specific problems encountered in the reports, along with the corresponding cleaning steps we implemented:

**1. Misaligned Headers**

* **Problem:** The header row, which typically contains sector names or year labels, was sometimes misaligned, especially when certain headers like "SECTORES ECON?MICOS" were incorrectly placed.
* **Solution:** We used the swap_nan_se() function to correct misalignments, ensuring the header "SECTORES ECON?MICOS" is placed in the correct column.

> d = swap_nan_se(d)

**2. Mixed Header Patterns**

* **Problem:** Some columns had combined headers, such as "Sector. Subsector," leading to confusion when analyzing data.
* **Solution:** The split_column_by_pattern() function was used to split these combined headers into separate, meaningful columns for easier analysis.

> d = split_column_by_pattern(d)

**3. Missing or Irregular Year Labels**

* **Problem:** Year labels (e.g., "2019", "2020") were either missing or misaligned across columns.
* **Solution:** We implemented the find_year_column() function to automatically detect and correct year columns, ensuring consistency across the data.

> d = find_year_column(d)

**4. Extra or Irrelevant Rows and Columns**

* **Problem:** Some tables contained rows or columns with redundant or irrelevant data (such as placeholders or completely missing values).
* **Solution:** We used functions like drop_nan_rows(), drop_nan_columns(), and drop_rare_caracter_row() to remove these unwanted entries.

> d = drop_nan_rows(d)
> d = drop_nan_columns(d)
> d = drop_rare_caracter_row(d)

**5. Mixed Numeric and Text Values**

* **Problem:** Some columns contained mixed content, with text and numeric values in the same column (e.g., "Var. %").
* **Solution:** The separate_text_digits() function was used to split the mixed content into separate numeric and text values, making the data easier to analyze.

> d = separate_text_digits(d)

**6. Formatting and Naming Inconsistencies**

* **Problem:** Sector names and labels had inconsistencies, especially between Spanish and English versions of terms like "services" and "mining."
* **Solution:** We standardized these terms using functions like replace_services() and replace_mineria() to harmonize the labels across all reports.

> d = replace_services(d)
> d = replace_mineria(d)
> d = replace_mining(d)

Final cleaning:

> d = clean_columns_values(d)
> d = convert_float(d)
> d = rounding_values(d, decimals=1)


### Cleaning Process: Code Walkthrough

The cleaning logic lives in peru_gdp_rtd.cleaners and is applied by the vintage builders.


#### Table 1: Monthly data into row-based vintage format

We use build_table_1_vintages to process old CSVs and shortened PDFs into vintage-format files stored in data/input/table_1/.


In [ ]:
build_table_1_vintages(
    old_csv_folder=str(settings.paths.old_weekly_reports),
    new_pdf_folder=str(settings.paths.pdf_input),
    output_folder=str(Path(settings.paths.data_input) / "table_1"),
    pipeline_version=settings.project["version"],
    persist_format=settings.features.persist_format,
    force=False,
)


In [ ]:
table1_dir = Path(settings.paths.data_input) / "table_1"
table1_files = sorted(table1_dir.rglob(f"*.{settings.features.persist_format}"))
table1_files[:5]


In [ ]:
if table1_files:
    sample = table1_files[0]
    df = pd.read_parquet(sample) if sample.suffix == ".parquet" else pd.read_csv(sample)
    df.head()


In this step, we build Table 1 vintages with build_table_1_vintages and inspect the output files saved in data/input/table_1/.


**Checking the cleaning version out**

In [ ]:
len(table1_files)


In [ ]:
if table1_files:
    df.columns[:10]


#### Table 2: Quarterly and annual data into row-based vintage format

We use build_table_2_vintages to process old CSVs and shortened PDFs into vintage-format files stored in data/input/table_2/.


In [ ]:
build_table_2_vintages(
    old_csv_folder=str(settings.paths.old_weekly_reports),
    new_pdf_folder=str(settings.paths.pdf_input),
    output_folder=str(Path(settings.paths.data_input) / "table_2"),
    pipeline_version=settings.project["version"],
    persist_format=settings.features.persist_format,
    force=False,
)


In [ ]:
table2_dir = Path(settings.paths.data_input) / "table_2"
table2_files = sorted(table2_dir.rglob(f"*.{settings.features.persist_format}"))
table2_files[:5]


**Checking the cleaning version out**

In [ ]:
if table2_files:
    sample = table2_files[0]
    df2 = pd.read_parquet(sample) if sample.suffix == ".parquet" else pd.read_csv(sample)
    df2.head()


In [ ]:
if table2_files:
    df2.columns[:10]


#### Key takeaways

- Tables are extracted and cleaned using tabula-py and custom cleaners.
- Vintages are written to data/input/table_1/ and data/input/table_2/.
- The output format is consistent across old and new sources.


> 🚀 **Next Steps:** With the data now cleaned and formatted, we can proceed to the next steps of building the real-time GDP dataset (RTD) by reshaping the tables and creating vintages for analysis. This will be covered in the following sections.

---

## Step 4: Concatenate RTD across years by frequency


---

This step concatenates all vintages into unified RTDs. Outputs are written to data/output/vintages/ using the configured persist_format (csv or parquet).


### Run the code to concatenate RTD

We use concatenate_table_1 (monthly) and concatenate_table_2 (quarterly/annual).


#### Table 1: Concatenate monthly data


In [ ]:
concatenated_1 = concatenate_table_1(
    input_data_subfolder=str(settings.paths.data_input),
    persist=True,
    persist_folder=str(settings.paths.vintages),
    csv_file_label=settings.output_files["monthly_rtd"],
    persist_format=settings.features.persist_format,
    force=False,
)


In [ ]:
# Check the first 10 rows of the concatenated data
concatenated_1.head(10)

#### Table 2: Concatenate quarterly and annual data


In [ ]:
concatenated_2 = concatenate_table_2(
    input_data_subfolder=str(settings.paths.data_input),
    persist=True,
    persist_folder=str(settings.paths.vintages),
    csv_file_label=settings.output_files["quarterly_annual_rtd"],
    persist_format=settings.features.persist_format,
    force=False,
)


In [ ]:
# Check the first 10 rows of the concatenated data
concatenated_2.head(10)

These functions will load the raw data from each year, concatenate it vertically, and return a unified DataFrame with the full RTD.

> ❗❗ **Disclaimer:** If compared with the tables displayed in the supplemental document, the concatenated tables above have been transposed to compactly save the dataset. This approach avoids creating excessively long datasets with too many columns, which could be cumbersome for most software when saved. Transposing the tables results in the same structure as the ones in the supplemental material

#### Key takeaways

- Concatenation aligns target periods and stacks vintages across years.
- Outputs are saved in data/output/vintages/.
- Summaries report processed files and final shape.


> 🚀 **Next Steps:** Once the data has been concatenated, we can proceed to the next steps in building the real-time GDP dataset (RTD). This involves reshaping the data and creating vintages for further analysis, which will be covered in the upcoming sections.

---

## Step 5: Metadata and benchmarks


---

This step updates metadata and generates base-year and benchmark datasets.

It includes:
1. Updating metadata in metadata/wr_metadata.csv.
2. Applying base-year sentinel adjustments to RTDs.
3. Generating benchmark datasets for monthly and quarterly RTDs (including base-year adjusted versions).


**Example: Define a List of Base Years**

In [ ]:
base_year_list = [
    {"year": by.year, "wr": by.wr, "base_year": by.base_year}
    for by in settings.metadata.base_years
]
base_year_list


The runner below updates metadata

In [ ]:
updated_df = update_metadata(
    metadata_folder=str(settings.paths.metadata),
    input_pdf_folder=str(settings.paths.pdf_input),
    wr_metadata_csv=settings.metadata.filename,
    base_year_list=base_year_list,
    force=False,
)


After updating the metadata, you can inspect the last few rows to verify the changes and ensure the revisions were applied correctly.

In [ ]:
updated_df.iloc[-10:]   # last 5 rows

### Generate base-year-adjusted RTDs

Apply a sentinel value to vintages affected by base-year changes.


**Example: Apply Base-Year Sentinel**

In [ ]:
base_year_list_2 = list(settings.benchmark.base_year_periods)


The `apply_base_year_sentinel` function applies a sentinel value (e.g., `-999999.0`) to data that is affected by a base-year change. This ensures that the affected data is marked as invalid, making it clear when and where the base-year changes occurred.

In [ ]:
adjusted_rtd = apply_base_year_sentinel(
    base_year_vintages=base_year_list_2,
    sentinel=settings.benchmark.sentinel_value,
    output_data_subfolder=str(settings.paths.vintages),
    csv_file_labels=[
        settings.output_files["monthly_rtd"],
        settings.output_files["quarterly_annual_rtd"],
    ],
    persist_format=settings.features.persist_format,
    force=False,
)


In [ ]:
adjusted_monthly_rtd = adjusted_rtd.get("by_adjusted_monthly_gdp_rtd")
adjusted_quarterly_rtd = adjusted_rtd.get("by_adjusted_quarterly_annual_gdp_rtd")


### Generate benchmark RTDs

Apply benchmark revision mapping using metadata/wr_metadata.csv.


**Example: Generate Benchmark RTDs**

In [ ]:
csv_file_labels = [
    settings.output_files["monthly_rtd"],
    settings.output_files["quarterly_annual_rtd"],
]
benchmark_dataset_labels = [
    settings.output_files["monthly_benchmark"],
    settings.output_files["quarterly_benchmark"],
]

csv_file_labels_adjusted = [
    settings.output_files["by_adjusted_monthly"],
    settings.output_files["by_adjusted_quarterly"],
]
benchmark_dataset_labels_adjusted = [
    settings.output_files["by_adjusted_monthly_benchmark"],
    settings.output_files["by_adjusted_quarterly_benchmark"],
]


In [ ]:
wr_metadata_csv = settings.metadata.filename


The `convert_to_benchmark_dataset` function applies the benchmark revision procedure to the real-time GDP data, ensuring that revisions are consistent with the latest updates from the statistical agencies.

In [ ]:
processed_datasets = convert_to_benchmark_dataset(
    output_data_subfolder=str(settings.paths.vintages),
    csv_file_labels=csv_file_labels,
    metadata_folder=str(settings.paths.metadata),
    wr_metadata_csv=wr_metadata_csv,
    benchmark_dataset_labels=benchmark_dataset_labels,
    persist_format=settings.features.persist_format,
    force=False,
)

processed_datasets_adjusted = convert_to_benchmark_dataset(
    output_data_subfolder=str(settings.paths.vintages),
    csv_file_labels=csv_file_labels_adjusted,
    metadata_folder=str(settings.paths.metadata),
    wr_metadata_csv=wr_metadata_csv,
    benchmark_dataset_labels=benchmark_dataset_labels_adjusted,
    persist_format=settings.features.persist_format,
    force=False,
)


In [ ]:
processed_datasets.keys(), processed_datasets_adjusted.keys()


In [ ]:
processed_datasets['monthly_gdp_benchmark']

### Key takeaways

- Metadata is refreshed from shortened PDFs.
- Base-year adjustments and benchmarks are saved to data/output/vintages/.


> 🚀 **Next Steps:** With the updated metadata and adjusted RTDs, we can proceed to the next steps in building the real-time GDP dataset (RTD). These steps will include reshaping the tables and creating vintages for in-depth analysis of GDP growth and revisions over time.

---

## Step 6: Releases


---

This section is responsible for converting Real-Time GDP (RTD) datasets into releases datasets. The releases dataset is crucial for tracking and analyzing the sequence of GDP revisions. By restructuring the data into a release-based format, we can better map the evolution of GDP estimates in terms of "releases", helping to capture changes and dependence patterns in the statistical analysis.

In this section, we will:

* Convert raw RTD data into release datasets.
* Align non-NaN values for each industry and vintage. The first release of all target periods aligns in the first row, and so on.
* Organize the data by release sequence for each industry and target period.

---

🛠️ Converting RTD to Releases Dataset

The `convert_to_releases_dataset` function is designed to transform the RTD data into a format that is structured by release sequence. This function processes each dataset, aligning the non-NaN values for every target period and each industry, while removing any invalid values due to base-year changes.

Key Steps in Conversion:

1. **File Validation:** Ensures that the input and output file lists match in length.
2. **Sorting and Grouping:** The data is sorted by industry, year, and month, ensuring chronological order.
3. **Aligning Non-NaN Values:** For each industry, the function aligns non-NaN values across the target periods (`tp_` columns), creating a sequence of releases.
4. **Removing Invalid Rows:** It drops rows where all target period columns are NaN, ensuring that only valid data is retained.
5. **Reorganization:** The dataset is pivoted, with each industry and release forming new columns.
6. **Final Output:** The dataset is saved into a CSV file for each industry and release sequence.

---

🔍 Data Processing Workflow

1. Input Data: We start with the RTD data, which is stored in CSV files. Each dataset corresponds to a specific frequency (monthly, quarterly, or annual) and includes data for different vintages.

2. Processing:

* Sorting: Data is sorted by industry, year, and month to ensure the chronological order of releases.
* Aligning Releases: Non-NaN values for each industry and vintage are aligned, ensuring that all releases are consistent and in the correct sequence.
* Pivoting: The data is then pivoted to arrange each industry’s releases in separate columns.

3. Output: The converted releases datasets are saved as new CSV files, each named according to its respective label (e.g., `monthly_gdp_releases.csv`, `quarterly_annual_gdp_releases.csv`).

---

🔄 Key Concepts and Terminology

* **Industry:** Represents the economic sector (e.g., manufacturing, agriculture) for which GDP growth rates are reported.
* **Vintage:** Refers to the specific release of GDP data for a given period (e.g., the first release, second release, etc.).
* **Release:** Each release refers to an updated estimate of GDP for a given target period. These releases are tracked sequentially (first, second, third, etc.) for each industry.
* **Target Period (tp_):** These are the columns representing GDP growth rates for specific periods (e.g., "tp_2021m01" for January 2021).

**Example: Convert RTD to Releases Dataset**

In [ ]:
csv_file_labels = [
    settings.output_files["monthly_rtd"],
    settings.output_files["quarterly_annual_rtd"],
    settings.output_files["by_adjusted_monthly"],
    settings.output_files["by_adjusted_quarterly"],
    settings.output_files["monthly_benchmark"],
    settings.output_files["quarterly_benchmark"],
    settings.output_files["by_adjusted_monthly_benchmark"],
    settings.output_files["by_adjusted_quarterly_benchmark"],
]
releases_dataset_labels = [
    settings.output_files["monthly_releases"],
    settings.output_files["quarterly_releases"],
    settings.output_files["by_adjusted_monthly_releases"],
    settings.output_files["by_adjusted_quarterly_releases"],
    settings.output_files["monthly_benchmark_releases"],
    settings.output_files["quarterly_benchmark_releases"],
    settings.output_files["by_adjusted_monthly_benchmark_releases"],
    settings.output_files["by_adjusted_quarterly_benchmark_releases"],
]


In [ ]:
releases_df = convert_to_releases_dataset(
    input_data_subfolder=str(settings.paths.vintages),
    output_data_subfolder=str(settings.paths.releases),
    csv_file_labels=csv_file_labels,
    releases_dataset_labels=releases_dataset_labels,
    persist_format=settings.features.persist_format,
    force=False,
)


After running the conversion, you can check the `releases_df` for specific datasets like "monthly_gdp_releases" to verify that the releases data has been processed and organized correctly.

In [ ]:
# Displaying the converted releases dataset for "monthly_gdp_releases"
releases_df["by_adjusted_monthly_gdp_releases"]

### Key takeaways

- Releases datasets align vintages into sequential releases by industry.
- Outputs are saved to data/output/releases/.


> 🚀 **Next Steps:** Now that the data has been converted into releases datasets, we can proceed with further analysis, including:
> * Revision analysis: Understanding how GDP estimates evolve over time.
> * Benchmark testing: Comparing the real-time dataset to benchmark revisions.

<div style="background:#3366FF; color:white; padding:12px; box-sizing:border-box; border-radius:4px;">
<b>🏁 The End</b>
</div>

---
---